# Comprensión del negocio

Escenario principal: siniestros con víctimas en Bogotá, 2018–2024. Actualización de alcance: 1 de octubre de 2026.

## Objetivo General

Desarrollar y evaluar retrospectivamente un prototipo académico de aprendizaje automático que asigne puntuaciones de priorización a combinaciones de **localidad × fecha × franja horaria**, según la ocurrencia o frecuencia elevada de siniestros con personas heridas o fallecidas registrados en Bogotá.

El propósito es estudiar patrones históricos y las posibilidades y limitaciones de su clasificación. El prototipo no acredita una reducción de siniestros ni está validado para asignar recursos institucionales o anticipar eventos en tiempo real.

**Resultado actual:** dataset preparado, modelos comparados, Random Forest ajustado como modelo principal, evaluación documentada, consulta retrospectiva por línea de comandos y **dashboard local funcional en Python Dash**. La aplicación permite explorar registros de 2018–2024 y consultar el modelo exclusivamente sobre casos preparados de 2023–2024. Está implementada para revisión académica; siguen pendientes la aceptación con usuarios y el despliegue público, si forma parte de la entrega. No se presenta como un producto validado para uso operativo.

El alcance y la versión del modelo se sustentan en el [registro oficial](../models/victimas/modelo_principal.json) y la [ficha técnica](../reports/modelo_principal/ficha_tecnica_modelo.md).

## Objetivos Específicos

1. Preparar los registros oficiales de Bogotá de 2018–2024, conservando los siniestros clasificados como **Con Heridos** o **Con Muertos** y construyendo un calendario completo por localidad y franja horaria.
2. Analizar patrones temporales, territoriales y por actor vial. El actor vial se utiliza únicamente en el análisis descriptivo; no es predictor ni criterio para generar alertas específicas por tipo de persona.
3. Construir variables de calendario e históricos anteriores al día evaluado, evitando usar el conteo del propio día como predictor.
4. Comparar Regresión Logística, Random Forest y XGBoost mediante cortes temporales. Ajustar la familia seleccionada y su umbral con información de 2018–2022, sin divisiones aleatorias que mezclen pasado y futuro.
5. Evaluar retrospectivamente sobre 2023–2024 la clasificación, el ordenamiento de casos, la calibración y los errores por localidad y franja, incluyendo referencias históricas sencillas y análisis de incertidumbre.
6. Ofrecer consultas reproducibles de los casos preparados de 2023–2024 mediante una interfaz local con mapa coroplético por localidad, filtros, trazabilidad y advertencias de interpretación, diferenciándolas del análisis descriptivo de 2018–2024. **Estado:** integración implementada en el dashboard Python Dash; aceptación con usuarios pendiente.

**Estado de selección:** el modelo principal es **Random Forest ajustado**, identificador `rf_victimas_bogota_v1.0`, con umbral de score **0,52**. La configuración base y el escenario original con todos los siniestros se conservan como referencias, no como modelo principal.

## Planteamiento del Problema

Los registros de siniestralidad permiten estudiar diferencias entre localidades, fechas y franjas horarias. El problema del proyecto consiste en determinar hasta qué punto el calendario, la localización y los antecedentes recientes permiten identificar combinaciones con una etiqueta positiva de ocurrencia o frecuencia elevada de siniestros con víctimas.

La observación no es una persona ni un accidente individual. Los registros se agregan por localidad, fecha y franja. Se cuentan **siniestros con víctimas, no personas heridas o fallecidas**; unir directamente las hojas de siniestros, vehículos y actores puede multiplicar eventos y alterar los conteos.

Durante el desarrollo se examinó el cambio de cobertura del escenario original que incluía siniestros de solo daños. El escenario principal se restringió a siniestros con víctimas durante todo 2018–2024 para mejorar la comparabilidad. Esta decisión no demuestra que la cobertura sea perfecta ni que todos los cambios entre años desaparezcan. Los análisis del escenario original se conservan como evidencia histórica.

La fuente congelada es la hoja `Siniestros` del Excel `base-anuario-de-siniestralidad-2024.xlsx`. El dataset principal agrupa **85.199 siniestros** en **204.560 observaciones**: 20 localidades × cuatro franjas × 2.557 días. Incluye combinaciones con cero eventos registrados; cero registros no garantiza que no haya ocurrido ningún evento.

El reto no se limita a obtener una métrica global: interesa medir falsas alertas, omisiones, diferencias territoriales y temporales, y el valor adicional frente a una referencia histórica simple. La evaluación actual no demuestra superioridad concluyente del RF frente a la tasa histórica por localidad, franja y día de semana.

Evidencias: [EDA con víctimas](../reports/eda_victimas/conclusion_ejecutiva.md), [evaluación ampliada](../reports/evaluation_victimas/conclusion_ejecutiva.md) y [procedencia del Excel](../data/raw/README.md). La ficha de procedencia mantiene como desconocida la fecha original de descarga y como pendiente la confirmación de la licencia específica.

## Pregunta de Investigación

**¿En qué medida las técnicas de aprendizaje automático, utilizando registros oficiales de Bogotá de 2018–2024, permiten identificar retrospectivamente combinaciones de localidad, fecha y franja horaria con ocurrencia o frecuencia elevada de siniestros con víctimas respecto de su historial, y qué valor aportan frente a referencias históricas sencillas?**

La pregunta se evalúa sobre la etiqueta definida en este notebook. No equivale a estimar la probabilidad individual de sufrir un accidente, demostrar causalidad ni validar una intervención de seguridad vial.

## ¿Qué se quiere predecir o analizar?

### Unidad de análisis y etiqueta

**Unidad:** localidad × fecha × franja horaria. Las franjas son Madrugada [00:00, 06:00), Mañana [06:00, 12:00), Tarde [12:00, 18:00) y Noche [18:00, 24:00).

La variable binaria `Alto_Riesgo` se define así:

`Alto_Riesgo = 1 si Num_Accidentes > cuantil 2/3 del conteo de su localidad–franja en entrenamiento; 0 en otro caso.`

Para el modelo final, el cuantil se calcula únicamente con 2018–2022. En cada corte de validación se recalcula con el pasado disponible. El nombre de la etiqueta se conserva por trazabilidad, pero su interpretación depende del grupo:

| Umbral histórico del conteo | Grupos localidad–franja | Significado de la etiqueta positiva |
| --- | ---: | --- |
| 0 | 49 | Al menos un siniestro con víctima |
| 1 | 28 | Dos o más siniestros con víctimas |
| 2 | 3 | Tres o más siniestros con víctimas |

Por tanto, en 49 de 80 grupos la tarea es de ocurrencia; no debe describirse como detección de frecuencia excepcional en todos los grupos. La etiqueta tampoco mide gravedad individual ni tasas ajustadas por exposición al tránsito.

### Predictores y antecedentes disponibles

Se usan 12 variables:

- Territorio y calendario: `Localidad`, `Franja_Horaria`, `Dia_Semana`, `Mes`, `Es_Fin_de_Semana` y `Es_Festivo`.
- Históricos: `Accidentes_Prom_7d`, `Accidentes_Prom_30d` y `Accidentes_Semana_Anterior`.
- Tres indicadores de ausencia de historial, con prefijo `Sin_Historial_` para cada variable histórica.

Los promedios excluyen el día evaluado y el rezago semanal usa el conteo de siete días antes, siempre dentro de la misma localidad y franja. Los faltantes iniciales se rellenan con cero y se identifican con sus indicadores. Ni `Num_Accidentes`, ni `Alto_Riesgo`, ni `Periodo` ni la fecha completa entran directamente al clasificador.

La evaluación utiliza los conteos anteriores observados; no simula varios días futuros sin actualizar antecedentes ni acredita que esos registros estén disponibles a tiempo para una operación real.

### Validación temporal y modelo principal

El entrenamiento final utiliza **2018–2022** y la evaluación retrospectiva **2023–2024**. La selección se realiza con cortes expansivos:

| Entrenamiento del corte | Validación |
| --- | --- |
| 2018–2019 | 2020 |
| 2018–2020 | 2021 |
| 2018–2021 | 2022 |

En cada corte se ajustan etiqueta, preprocesamiento y modelo con el pasado disponible. 04B compara las familias y 04C evalúa ocho configuraciones de Random Forest, seleccionando la mayor Average Precision media. Luego elige el umbral que maximiza F1 sobre las predicciones fuera de muestra agregadas de 2020–2022.

La configuración principal tiene **300 árboles**, profundidad sin límite explícito, mínimo **20 observaciones por hoja**, `max_features='sqrt'`, `class_weight='balanced'` y semilla **42**. No se afirma que sea un óptimo global.

Las validaciones se reutilizan para configuración y umbral: sus métricas son de selección, no una estimación independiente. Aunque 2023–2024 no se usa para calcular esa búsqueda, ya fue inspeccionado durante el desarrollo y no constituye una prueba prospectiva intacta.

### Salida e interpretación

El modelo produce `Score_Priorizacion` y activa `Alerta_Modelo = 1` cuando **score ≥ 0,52**. Este umbral del score es distinto del cuantil utilizado para construir la etiqueta observada.

El score es una **puntuación relativa no calibrada**, no una probabilidad literal de accidente. La consulta implementada selecciona una observación ya preparada de 2023–2024 y aplica el modelo cerrado; rechaza fechas fuera de ese periodo.

### Análisis descriptivo y límites

El EDA comprende territorio, calendario, franjas y actores viales. Los análisis espaciales exploratorios no cambian la resolución predictiva: las alertas corresponden exclusivamente a localidades, no a calles o coordenadas.

El prototipo no genera predicciones por actor vial, no incorpora clima ni tráfico en tiempo real, no pronostica fechas nuevas y no estima efectos causales. La consulta actual no modifica datos ni reentrena el modelo.

## Beneficios del Modelo

### Aporte académico y trazabilidad

El proyecto permite estudiar una formulación reproducible de clasificación temporal con registros oficiales, conservar las decisiones de preparación y comparar algoritmos con referencias sencillas. Su contribución comprobada es metodológica y experimental; no se ha medido una reducción de siniestros ni un ahorro económico.

### Exploración de resultados y errores

La consulta retrospectiva permite examinar puntuaciones, alertas y antecedentes de casos históricos. La evaluación identifica dónde se concentran aciertos, falsas alertas y omisiones, y la importancia por permutación describe dependencia predictiva, no causalidad.

### Criterios de evaluación y límites de utilidad

Se consideran F1, precisión, recall, Average Precision, AUC-ROC, Brier, matrices de confusión, análisis por subgrupo y bootstrap temporal. La evaluación del RF ajustado reproduce **F1 0,4058**, **AUC-ROC 0,6933**, **Average Precision 0,3098**, **precisión 0,2941** y **recall 0,6544**.

- El Brier **0,2221** es peor que la referencia constante de entrenamiento **0,1544**: no se respalda interpretar el score como probabilidad calibrada.
- La referencia histórica localidad–franja–día de semana alcanza AP **0,3087** y AUC-ROC **0,6931**. Los intervalos exploratorios de la diferencia frente al RF incluyen cero; no demuestran superioridad concluyente ni equivalencia.
- El desempeño global no garantiza detección homogénea. Por ejemplo, Candelaria tiene 157 positivos y ninguno detectado con el umbral cerrado.

Estos resultados justifican una herramienta de exploración académica con advertencias, no una recomendación automática de intervención. Antes de un uso operativo se necesitarían validación prospectiva, disponibilidad oportuna de antecedentes y criterios de costo y aceptación acordados.

### Usuario y estado de entrega

El usuario previsto es un analista, investigador o persona interesada en explorar los resultados históricos. El dashboard local de solo lectura ya está implementado en Python Dash. Reutiliza el pipeline registrado, conserva la resolución por localidad y separa descripción, consulta del modelo y evaluación.

| Sección implementada | Alcance |
| --- | --- |
| Resumen | Universo, indicadores y evolución de registros con víctimas |
| Consulta y mapa | Fecha de 2023–2024, localidad y franja; score, alerta, antecedentes y trazabilidad; mapa de las 20 localidades |
| Análisis histórico | Registros de 2018–2024 con filtros por año, localidad, franja y participación de actor |
| Evaluación | Métricas filtradas, calibración, curvas, errores y comparaciones globales identificadas como tales |
| Metodología | Unidad, etiqueta, validación temporal, fuente, modelo e interpretación |

La aplicación genera descargas JSON/CSV en memoria, sin cambiar datos ni reentrenar. El filtro de actor es exclusivamente descriptivo; las categorías pueden solaparse. El score no se presenta como probabilidad calibrada.

**Verificación disponible:** pruebas automáticas del dashboard, integración con las 58.480 decisiones conservadas y revisión en navegador de las cinco secciones, filtros, presentación móvil y descarga de consulta. Estas comprobaciones técnicas **no sustituyen** una prueba de aceptación con usuarios ni una auditoría completa de accesibilidad.

**Pendientes de entrega:** validar comprensión y uso con usuarios, completar comprobaciones de interfaz y accesibilidad y, si se decide publicar, definir alojamiento y operación. El servicio actual solo escucha en `127.0.0.1`; no hay despliegue público. La confirmación de las condiciones específicas de reutilización de la fuente sigue pendiente.

Evidencias de la interfaz: [guía técnica y de uso](../dashboard/README.md), [entrega local del dashboard](../reports/dashboard/ENTREGA_20261001.md) y [verificación de integración del dashboard](../reports/dashboard/validacion_final_20261001.json).

La reproducibilidad se respalda con pruebas de preparación y consulta, registro de versión, verificaciones de integridad y evidencias de ejecución aislada. Véanse la [ficha técnica](../reports/modelo_principal/ficha_tecnica_modelo.md), la [evaluación](../reports/evaluation_victimas/conclusion_ejecutiva.md) y el [cierre técnico](../reports/cierre_tecnico/INFORME_20260926.md).

## Estructura del proyecto

El recorrido vigente del escenario con víctimas es **02B → 03B → 04B → 04C → 05B → dashboard local**. Los notebooks originales se conservan como antecedentes; no deben confundirse con el modelo principal. El [catálogo de artefactos](../config/artefactos.json) identifica ambos escenarios.

```text
Siniestralidad_Vial/
├── data/
│   ├── raw/                       # Excel congelado y procedencia
│   ├── processed/                 # Dataset principal y escenario histórico
│   ├── reference/                 # Cartografía y procedencia
│   └── dashboard/                 # Agregados descriptivos y manifiesto de integridad
├── notebooks/
│   ├── 01_Comprension_del_Negocio.ipynb
│   ├── 02B_EDA_y_Analisis_Espacial_Con_Victimas.ipynb
│   ├── 03B_Preparacion_Datos_Con_Victimas.ipynb
│   ├── 04B_Modelado_Con_Victimas.ipynb
│   ├── 04C_Ajuste_Hiperparametros_Con_Victimas.ipynb
│   └── 05B_Evaluacion_Con_Victimas.ipynb
├── src/                           # Preparación, carga y consulta retrospectiva
├── scripts/                       # Verificación y reproducción aislada
├── tests/                         # Pruebas automáticas
├── models/victimas/                # Pipeline principal, registro y referencias
├── reports/                       # EDA, experimentos, evaluación y evidencias
├── config/artefactos.json          # Catálogo principal, base y legado
├── dashboard/                     # Aplicación local Python Dash implementada
│   ├── app.py                     # Interfaz, filtros y descargas
│   ├── data.py                    # Carga verificada y métricas
│   ├── charts.py                  # Gráficas y mapa
│   ├── assets/                    # Estilos adaptables
│   ├── README.md                  # Guía técnica y de uso
│   └── dist/                      # Maqueta estática anterior, no aplicación vigente
├── .github/workflows/pruebas.yml   # Pruebas en GitHub
├── README.md
└── requirements.txt
```

### Uso y comprobación

Desde la raíz del proyecto, con el entorno instalado:

```sh
python scripts/consultar_modelo.py --fecha 2023-01-01 --localidad "CANDELARIA" --franja "Noche"
python scripts/verificar_codigo.py
python scripts/verificar_codigo.py --integracion
python scripts/verificar_dashboard.py
```

Para iniciar la interfaz local:

```sh
python -m dashboard.app
```

Abrir <http://127.0.0.1:8050>. El proceso permanece activo hasta detenerlo con Ctrl+C. No es una dirección pública; instalación y diagnóstico en la [guía del dashboard](../dashboard/README.md).

La consulta devuelve el score, la alerta, el umbral, las variables de la observación y la trazabilidad del modelo. La interfaz reutiliza esta consulta y conserva la identidad, versión, periodo y advertencias en sus exportaciones del modelo. La verificación de integración requiere los archivos reales de Git LFS y comprueba fuente, preparación, predicciones y cálculos del dashboard sin reentrenar ni sobrescribir los artefactos oficiales.

Si se desea repetir la cadena con entrenamiento, debe utilizarse `scripts/reproducir_cadena_aislada.py` para trabajar en una copia independiente. No reejecutar los notebooks de entrenamiento sobre el cierre oficial.

Este notebook contiene únicamente documentación; no ejecuta preparación, entrenamiento ni evaluación. Los resultados citados proceden de las evidencias enlazadas.
